# Notebook 01: Demostración de Data Leakage

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


# 1. Cargar datos
df = pd.read_csv("heart.csv")

print("Columnas del dataset:")
print(df.columns)

display(df.head())


# 2. Definir variable objetivo
# En este dataset normalmente la variable objetivo es HeartDisease
target_col = "HeartDisease"

X = df.drop(target_col, axis=1)
y = df[target_col]

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("\nColumnas numéricas:")
print(list(numeric_features))

print("\nColumnas categóricas:")
print(list(categorical_features))

# Preprocesamiento correcto
preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


# 3. Ejemplo CON fuga de datos
# Copia de X para crear una variable artificial con fuga
X_leaky = X.copy()

np.random.seed(0)
X_leaky["leaky_feature"] = y + np.random.normal(0, 0.01, size=len(y))

# Recalcular columnas porque agregamos leaky_feature
numeric_features_l = X_leaky.select_dtypes(include=["int64", "float64"]).columns
categorical_features_l = X_leaky.select_dtypes(include=["object", "category"]).columns

preprocessor_l = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features_l),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_l)
    ]
)

# Aquí hacemos un flujo con fuga porque incluimos una variable que contiene información de y
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

pipe_l = Pipeline([
    ("preprocessor", preprocessor_l),
    ("svc", SVC(probability=True))
])

param_grid_l = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": [0.01, 0.1]
}

grid_l = GridSearchCV(
    pipe_l,
    param_grid_l,
    cv=5,
    scoring="roc_auc"
)

grid_l.fit(X_train_l, y_train_l)

auc_l = roc_auc_score(
    y_test_l,
    grid_l.predict_proba(X_test_l)[:, 1]
)


# 4. Flujo correcto SIN fuga de datos
# Aquí usamos X original, SIN leaky_feature
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("svc", SVC(probability=True))
])

param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": [0.01, 0.1]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

auc = roc_auc_score(
    y_test,
    grid.predict_proba(X_test)[:, 1]
)


# 5. Resultados
print(f"AUC con fuga: {auc_l:.3f}")
print(f"AUC sin fuga: {auc:.3f}")

print("\nMejores hiperparámetros con fuga:")
print(grid_l.best_params_)

print("\nMejores hiperparámetros sin fuga:")
print(grid.best_params_)

Columnas del dataset:
Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0



Columnas numéricas:
['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

Columnas categóricas:
['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
AUC con fuga: 1.000
AUC sin fuga: 0.928

Mejores hiperparámetros con fuga:
{'svc__C': 0.1, 'svc__gamma': 0.1}

Mejores hiperparámetros sin fuga:
{'svc__C': 1, 'svc__gamma': 0.1}
